# Notebook 02 — Non-Destructive Arabic Data Preprocessing

This notebook creates a clean, model-ready **derived dataset** from the original source. It preserves every original field and adds cleaned/derived fields plus quality flags. No source row is silently deleted.

In [ ]:
from pathlib import Path
import re, json, unicodedata
import numpy as np
import pandas as pd

DATA_PATH=Path('data/Hotel Arabic-Reviews Dataset.txt')
if not DATA_PATH.exists(): DATA_PATH=Path('/home/ubuntu/work_sentiment/sentiment_analysis_submission/data/Hotel Arabic-Reviews Dataset.txt')
OUT=Path('data/processed'); OUT.mkdir(parents=True,exist_ok=True)
df=pd.read_csv(DATA_PATH,sep='\t',encoding='utf-16',dtype=str,keep_default_na=False)
ORIGINAL_COLUMNS=list(df.columns)
print('Loaded:',df.shape)

## Processing policy — full cleaning for modeling

The source remains untouched. The notebook creates a traceable full dataset and a separate `model_ready` dataset:

1. Preserve all original columns under `original_*`.
2. Convert `-`, blank, and null-like tokens to explicit `Unknown` values in model metadata.
3. Normalize whitespace, Arabic orthography, punctuation, URLs, diacritics, and tatweel in derived text.
4. Parse `nights` into a numeric field; unresolved values become `Unknown`/`NaN` with a flag.
5. Correct the 72 room-like `user type` values only when the room field is missing; otherwise mark them and use `Unknown` for the model metadata.
6. Remove exact duplicate normalized reviews from the model-ready set, retaining the first occurrence and logging the number removed.
7. Quarantine empty/too-short reviews instead of silently deleting them.
8. Keep all original rows in the full processed audit file; only the separate model-ready file is filtered.

In [ ]:
missing_tokens={'','-','--','N/A','NA','n/a','null','None'}
def clean_missing(s):
    s=s.astype(str).str.strip()
    return s.mask(s.isin(missing_tokens), pd.NA)

def normalize_arabic(text):
    if pd.isna(text): return pd.NA
    text=str(text)
    text=re.sub(r'https?://\S+|www\.\S+',' ',text)
    text=re.sub(r'[ًٌٍَُِّْـ]','',text)
    text=re.sub('[إأآا]','ا',text); text=re.sub('ى','ي',text); text=re.sub('ؤ','و',text); text=re.sub('ئ','ي',text)
    text=re.sub(r'[“”"،؛؟!,.():;\[\]{}<>_+=*#@/&%$^|~`\-]',' ',text)
    text=re.sub(r'\s+',' ',text).strip().lower()
    return text if text else pd.NA

def clean_metadata(s):
    if pd.isna(s): return pd.NA
    return re.sub(r'\s+',' ',str(s)).strip()

# Preserve originals exactly as strings.
for c in ORIGINAL_COLUMNS: df['original_'+c.replace(' ','_')]=df[c]
for c in ['Hotel name','user type','room type','nights']:
    df[c.replace(' ','_')+'_clean']=clean_missing(df[c]).map(clean_metadata)
df['review_clean']=df['review'].map(normalize_arabic)

In [ ]:
# Derive numeric rating and sentiment; original rating remains untouched.
df['rating_num']=pd.to_numeric(df['rating'],errors='coerce')
df['rating_valid']=df['rating_num'].between(1,5)
df['sentiment']=pd.cut(df['rating_num'],bins=[0,2,3,5],labels=['negative','neutral','positive'],include_lowest=True)

### Extracting nights

The source column is textual. We extract a numeric value where possible but retain the cleaned text and add a flag when the value is unknown or ambiguous.

In [ ]:
ARABIC_DIGITS=str.maketrans('٠١٢٣٤٥٦٧٨٩','0123456789')
word_numbers={'ليلة واحدة':1,'ليلة':1,'ليلتين':2,'ليلتان':2,'ليالي':None}
def extract_nights(x):
    if pd.isna(x): return np.nan
    s=str(x).translate(ARABIC_DIGITS).strip().lower()
    if s in missing_tokens: return np.nan
    m=re.search(r'\d+',s)
    if m: return float(m.group())
    if 'ليلتين' in s: return 2.0
    if 'ليلة واحدة' in s or s=='ليلة': return 1.0
    return np.nan
df['nights_num']=df['nights_clean'].map(extract_nights)
df['nights_missing']=df['nights_num'].isna()

In [ ]:
# Explicit quality flags. These make problems measurable instead of hiding them.
df['flag_review_empty']=df['review_clean'].isna() | (df['review_clean'].fillna('').str.len()==0)
df['flag_review_too_short']=df['review_clean'].fillna('').str.len()<3
df['flag_review_has_latin']=df['review'].str.contains(r'[A-Za-z]',regex=True,na=False)
df['flag_user_type_suspect']=df['user_type_clean'].fillna('').str.contains(r'غرفة|جناح|استوديو|شقة|سرير|توأم|ديلوكس|قياسية|سوبيريور',regex=True)
df['flag_missing_metadata']=df[['user_type_clean','room_type_clean','nights_clean']].isna().any(axis=1)
df['flag_duplicate_review']=df['review_clean'].duplicated(keep=False)
print('flag counts:')
display(df.filter(regex='^flag_|^rating_valid$').sum().sort_values(ascending=False).to_frame('count'))

## Output schema and validation

The processed output is a CSV derived from the source. It is not a replacement for the source file.

In [ ]:
processed_path=OUT/'hotel_reviews_processed.csv'
df.to_csv(processed_path,index=False,encoding='utf-8-sig')
manifest={
 'source':str(DATA_PATH),'output':str(processed_path),'source_rows':int(len(df)),'output_rows':int(len(df)),
 'original_columns':ORIGINAL_COLUMNS,'added_columns':[c for c in df.columns if c not in ORIGINAL_COLUMNS],
 'policy':'non-destructive derived dataset; original columns preserved with original_ prefix',
 'sentiment_mapping':{'1-2':'negative','3':'neutral','4-5':'positive'}
}
(OUT/'processing_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')
print('wrote',processed_path,'rows',len(df),'columns',len(df.columns))

In [ ]:
# Round-trip validation
check=pd.read_csv(processed_path,encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert len(check)==len(df)
assert all(c in check.columns for c in ORIGINAL_COLUMNS)
print('Round-trip validation passed.')
print('Sentiment distribution:'); display(df['sentiment'].value_counts(dropna=False).to_frame('count'))
print('Preview of derived fields:'); display(df[['review','review_clean','nights','nights_num','sentiment','flag_missing_metadata','flag_user_type_suspect']].head(10))

## Ready for modeling

The next modeling notebooks should use `data/processed/hotel_reviews_processed.csv`, select `review_clean` or `review` according to the model, use `sentiment` as the target, and exclude all identifiers and post-outcome leakage. The source remains available for audit and traceability.

## Final full cleaning and model-ready export

The following cell applies the documented decisions. It does not overwrite the full processed dataset. It creates a clean modeling table and a quarantine table with an explicit reason for every excluded row.

In [ ]:
# Resolve missing metadata and suspicious column placement for modeling only.
model_df=df.copy()
for c in ['Hotel_name_clean','user_type_clean','room_type_clean','nights_clean']:
    model_df[c]=model_df[c].fillna('Unknown').replace('', 'Unknown')

# Identify room-like user types. Correct only when the room field is missing/Unknown.
room_like=model_df['user_type_clean'].str.contains(r'غرفة|جناح|استوديو|شقة|سرير|توأم|ديلوكس|قياسية|سوبيريور',regex=True,na=False)
room_missing=model_df['room_type_clean'].eq('Unknown')
model_df['flag_user_type_corrected']=room_like & room_missing
model_df.loc[model_df['flag_user_type_corrected'],'room_type_clean']=model_df.loc[model_df['flag_user_type_corrected'],'user_type_clean']
# For remaining room-like values, prevent a misleading user-type feature.
model_df.loc[room_like,'user_type_clean']='Unknown'

# Normalize any unresolved numeric nights to a documented sentinel for non-text metadata.
model_df['nights_num_model']=model_df['nights_num'].fillna(-1).astype(float)
model_df['nights_known']=model_df['nights_num'].notna()

# Keep valid ratings only; the audit has shown all source ratings are valid.
model_df['exclude_reason']=''
model_df.loc[~model_df['rating_valid'],'exclude_reason']='invalid_rating'
model_df.loc[model_df['flag_review_empty'],'exclude_reason']='empty_after_normalization'
model_df.loc[(model_df['exclude_reason']=='') & model_df['flag_review_too_short'],'exclude_reason']='too_short_after_normalization'

# Deduplicate only within the clean modeling view; preserve first row and log exclusions.
seen=model_df['review_clean'].duplicated(keep='first')
model_df.loc[(model_df['exclude_reason']=='') & seen,'exclude_reason']='duplicate_normalized_review'
model_df['is_model_ready']=model_df['exclude_reason'].eq('')
quarantine=model_df.loc[~model_df['is_model_ready']].copy()
model_ready=model_df.loc[model_df['is_model_ready']].copy()

# Remove internal working flags from the final feature table only after logging them.
model_ready_path=OUT/'hotel_reviews_model_ready.csv'
quarantine_path=OUT/'hotel_reviews_quarantine.csv'
model_ready.to_csv(model_ready_path,index=False,encoding='utf-8-sig')
quarantine.to_csv(quarantine_path,index=False,encoding='utf-8-sig')
cleaning_log={
 'input_rows':int(len(df)), 'model_ready_rows':int(len(model_ready)), 'quarantine_rows':int(len(quarantine)),
 'excluded_by_reason':quarantine['exclude_reason'].value_counts().to_dict(),
 'corrected_room_like_user_values':int(model_df['flag_user_type_corrected'].sum()),
 'duplicate_normalized_reviews_removed':int((quarantine['exclude_reason']=='duplicate_normalized_review').sum()),
 'policy':'source unchanged; full processed file retained; model-ready is a derived filtered view'
}
(OUT/'model_ready_cleaning_log.json').write_text(json.dumps(cleaning_log,ensure_ascii=False,indent=2),encoding='utf-8')
print(json.dumps(cleaning_log,ensure_ascii=False,indent=2))


In [ ]:
# Final validation: model-ready has no unresolved review/label problems.
check=pd.read_csv(model_ready_path,encoding='utf-8-sig',dtype=str,keep_default_na=False)
assert len(check)==len(model_ready)
assert (check['review_clean'].str.len()>=3).all()
assert check['rating_valid'].astype(str).str.lower().eq('true').all()
assert check['exclude_reason'].eq('').all()
print('MODEL-READY VALIDATION PASSED')
print('rows:',len(check),'columns:',len(check.columns))
print('remaining missing metadata:',(check[['Hotel_name_clean','user_type_clean','room_type_clean']]=='').sum().sum())
print('sentiment:',check['sentiment'].value_counts().to_dict())


## Final decision

`hotel_reviews_model_ready.csv` is the file to use for ML/DL/Transformer experiments. `hotel_reviews_processed.csv` remains the complete traceable derived dataset, and `hotel_reviews_quarantine.csv` contains every excluded row with an explicit reason. The raw source is never overwritten.